In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("  Factor Research Module Initialized")

## 1. Generate Multi-Asset Universe

In [ ]:
# Generate synthetic returns for multiple assets
np.random.seed(42)
n_assets = 50
n_days = 1000
dates = pd.date_range(start='2020-01-01', periods=n_days, freq='D')

# Generate factor returns (market, size, value, momentum)
market_factor = np.random.normal(0.0005, 0.015, n_days)
size_factor = np.random.normal(0.0002, 0.01, n_days)
value_factor = np.random.normal(0.0003, 0.012, n_days)
momentum_factor = np.random.normal(0.0001, 0.008, n_days)

# Create factor DataFrame
factors = pd.DataFrame({
    'Market': market_factor,
    'Size': size_factor,
    'Value': value_factor,
    'Momentum': momentum_factor
}, index=dates)

# Generate asset returns with different factor loadings
returns_data = {}
factor_loadings = {}

for i in range(n_assets):
    # Random factor exposures (betas)
    beta_market = np.random.uniform(0.5, 1.5)
    beta_size = np.random.uniform(-0.5, 0.5)
    beta_value = np.random.uniform(-0.5, 0.5)
    beta_momentum = np.random.uniform(-0.3, 0.3)
    
    # Store loadings
    factor_loadings[f'Asset_{i+1}'] = {
        'Market': beta_market,
        'Size': beta_size,
        'Value': beta_value,
        'Momentum': beta_momentum
    }
    
    # Generate returns using factor model
    idiosyncratic = np.random.normal(0, 0.01, n_days)
    asset_returns = (
        beta_market * market_factor +
        beta_size * size_factor +
        beta_value * value_factor +
        beta_momentum * momentum_factor +
        idiosyncratic
    )
    
    returns_data[f'Asset_{i+1}'] = asset_returns

# Create returns DataFrame
returns = pd.DataFrame(returns_data, index=dates)

# Calculate prices
prices = 100 * (1 + returns).cumprod()

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Asset prices
axes[0].plot(prices.index, prices.iloc[:, :10], alpha=0.7, linewidth=1)
axes[0].set_ylabel('Price')
axes[0].set_title('Sample Asset Prices (First 10 Assets)')
axes[0].grid(True, alpha=0.3)

# Factor returns
cumulative_factors = (1 + factors).cumprod()
for col in cumulative_factors.columns:
    axes[1].plot(cumulative_factors.index, cumulative_factors[col], linewidth=2, label=col)
axes[1].set_ylabel('Cumulative Return')
axes[1].set_title('Factor Performance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Correlation matrix
corr_matrix = returns.iloc[:, :20].corr()
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, ax=axes[2], 
           xticklabels=False, yticklabels=False, cbar_kws={'label': 'Correlation'})
axes[2].set_title('Asset Return Correlations (First 20 Assets)')

plt.tight_layout()
plt.show()

print(f"Generated {n_assets} assets over {n_days} days")
print(f"\nFactor Statistics:")
print(factors.describe().round(6))

## 2. Fama-French Factor Model

In [ ]:
def fama_french_regression(asset_returns, factor_returns):
    """Run Fama-French three-factor regression"""
    model = LinearRegression()
    X = factor_returns[['Market', 'Size', 'Value']].values
    y = asset_returns.values
    
    model.fit(X, y)
    
    # Calculate alpha (intercept) and betas (coefficients)
    alpha = model.intercept_
    betas = model.coef_
    
    # Calculate R-squared
    predictions = model.predict(X)
    ss_total = np.sum((y - y.mean())**2)
    ss_residual = np.sum((y - predictions)**2)
    r_squared = 1 - (ss_residual / ss_total)
    
    # Calculate residuals
    residuals = y - predictions
    
    return {
        'alpha': alpha,
        'beta_market': betas[0],
        'beta_size': betas[1],
        'beta_value': betas[2],
        'r_squared': r_squared,
        'residuals': residuals
    }

# Run regression for all assets
ff_results = {}
for asset in returns.columns:
    ff_results[asset] = fama_french_regression(returns[asset], factors)

# Convert to DataFrame
ff_df = pd.DataFrame(ff_results).T

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Alpha distribution
axes[0, 0].hist(ff_df['alpha'] * 252 * 100, bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Annualized Alpha (%)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Alpha Distribution')
axes[0, 0].grid(True, alpha=0.3)

# Market beta distribution
axes[0, 1].hist(ff_df['beta_market'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].axvline(x=1, color='red', linestyle='--', linewidth=2, label='Market Beta = 1')
axes[0, 1].set_xlabel('Market Beta')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Market Beta Distribution')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# R-squared distribution
axes[1, 0].hist(ff_df['r_squared'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_xlabel('R-squared')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Model Fit (R-squared) Distribution')
axes[1, 0].grid(True, alpha=0.3)

# Factor loadings scatter
axes[1, 1].scatter(ff_df['beta_size'], ff_df['beta_value'], alpha=0.6, s=50)
axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[1, 1].axvline(x=0, color='black', linestyle='--', alpha=0.3)
axes[1, 1].set_xlabel('Size Beta')
axes[1, 1].set_ylabel('Value Beta')
axes[1, 1].set_title('Size vs Value Factor Exposures')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Top alpha generators
top_alpha = ff_df['alpha'].nlargest(5) * 252 * 100
print("\n=== Top 5 Alpha Generators (Annualized) ===")
for asset, alpha in top_alpha.items():
    print(f"{asset}: {alpha:.2f}%")

print(f"\nAverage R-squared: {ff_df['r_squared'].mean():.4f}")
print(f"Average Market Beta: {ff_df['beta_market'].mean():.4f}")

## 3. Principal Component Analysis (PCA)

In [ ]:
# Standardize returns
scaler = StandardScaler()
returns_scaled = scaler.fit_transform(returns)

# Apply PCA
n_components = 10
pca = PCA(n_components=n_components)
principal_components = pca.fit_transform(returns_scaled)

# Create PC DataFrame
pc_df = pd.DataFrame(
    principal_components,
    columns=[f'PC{i+1}' for i in range(n_components)],
    index=returns.index
)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Explained variance
explained_var = pca.explained_variance_ratio_ * 100
cumulative_var = np.cumsum(explained_var)

axes[0, 0].bar(range(1, n_components + 1), explained_var, alpha=0.7, edgecolor='black')
axes[0, 0].plot(range(1, n_components + 1), cumulative_var, 'ro-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Principal Component')
axes[0, 0].set_ylabel('Explained Variance (%)')
axes[0, 0].set_title('PCA - Explained Variance by Component')
axes[0, 0].legend(['Cumulative', 'Individual'])
axes[0, 0].grid(True, alpha=0.3)

# First 3 PCs over time
for i in range(3):
    axes[0, 1].plot(pc_df.index, pc_df[f'PC{i+1}'], linewidth=1.5, label=f'PC{i+1}')
axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[0, 1].set_ylabel('Component Value')
axes[0, 1].set_title('First 3 Principal Components Over Time')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# PC1 vs PC2 scatter
axes[1, 0].scatter(pc_df['PC1'], pc_df['PC2'], alpha=0.5, s=20)
axes[1, 0].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[1, 0].axvline(x=0, color='black', linestyle='--', alpha=0.3)
axes[1, 0].set_xlabel('PC1')
axes[1, 0].set_ylabel('PC2')
axes[1, 0].set_title('PC1 vs PC2 Scatter')
axes[1, 0].grid(True, alpha=0.3)

# Factor loadings heatmap for first 3 PCs
loadings = pca.components_[:3, :20]  # First 3 PCs, first 20 assets
sns.heatmap(loadings, cmap='coolwarm', center=0, ax=axes[1, 1],
           xticklabels=returns.columns[:20], yticklabels=[f'PC{i+1}' for i in range(3)],
           cbar_kws={'label': 'Loading'})
axes[1, 1].set_title('Factor Loadings (First 3 PCs, First 20 Assets)')

plt.tight_layout()
plt.show()

print(f"\n=== PCA Summary ===")
print(f"Total variance explained by {n_components} components: {cumulative_var[-1]:.2f}%")
print(f"Variance explained by PC1: {explained_var[0]:.2f}%")
print(f"Variance explained by PC1-PC3: {cumulative_var[2]:.2f}%")

## 4. Custom Factor Construction

In [ ]:
def construct_momentum_factor(returns, lookback=60):
    """Construct momentum factor using cross-sectional rankings"""
    # Calculate lookback returns
    lookback_returns = returns.rolling(window=lookback).apply(lambda x: (1 + x).prod() - 1)
    
    # Rank assets (cross-sectional)
    ranks = lookback_returns.rank(axis=1, pct=True)
    
    # Long-short portfolio: long top 30%, short bottom 30%
    long_positions = (ranks > 0.7).astype(float)
    short_positions = (ranks < 0.3).astype(float)
    
    # Normalize positions
    long_weight = long_positions / long_positions.sum(axis=1).replace(0, 1).values.reshape(-1, 1)
    short_weight = short_positions / short_positions.sum(axis=1).replace(0, 1).values.reshape(-1, 1)
    
    # Factor return = long portfolio return - short portfolio return
    factor_returns = (returns * (long_weight - short_weight)).sum(axis=1)
    
    return factor_returns

def construct_volatility_factor(returns, lookback=20):
    """Construct low-volatility factor"""
    # Calculate rolling volatility
    volatility = returns.rolling(window=lookback).std()
    
    # Rank assets (lower volatility = higher rank)
    ranks = volatility.rank(axis=1, pct=True, ascending=True)
    
    # Long-short portfolio
    long_positions = (ranks > 0.7).astype(float)
    short_positions = (ranks < 0.3).astype(float)
    
    # Normalize positions
    long_weight = long_positions / long_positions.sum(axis=1).replace(0, 1).values.reshape(-1, 1)
    short_weight = short_positions / short_positions.sum(axis=1).replace(0, 1).values.reshape(-1, 1)
    
    factor_returns = (returns * (long_weight - short_weight)).sum(axis=1)
    
    return factor_returns

def construct_quality_factor(returns, lookback=252):
    """Construct quality factor based on return stability"""
    # Calculate Sharpe ratio for each asset
    mean_returns = returns.rolling(window=lookback).mean()
    std_returns = returns.rolling(window=lookback).std()
    sharpe_ratios = mean_returns / std_returns
    
    # Rank assets
    ranks = sharpe_ratios.rank(axis=1, pct=True)
    
    # Long-short portfolio
    long_positions = (ranks > 0.7).astype(float)
    short_positions = (ranks < 0.3).astype(float)
    
    # Normalize positions
    long_weight = long_positions / long_positions.sum(axis=1).replace(0, 1).values.reshape(-1, 1)
    short_weight = short_positions / short_positions.sum(axis=1).replace(0, 1).values.reshape(-1, 1)
    
    factor_returns = (returns * (long_weight - short_weight)).sum(axis=1)
    
    return factor_returns

# Construct custom factors
momentum_factor = construct_momentum_factor(returns, lookback=60)
volatility_factor = construct_volatility_factor(returns, lookback=20)
quality_factor = construct_quality_factor(returns, lookback=252)

# Combine into DataFrame
custom_factors = pd.DataFrame({
    'Momentum': momentum_factor,
    'Low_Volatility': volatility_factor,
    'Quality': quality_factor
})

# Calculate cumulative returns
cumulative_custom = (1 + custom_factors).cumprod()

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Cumulative returns
for col in cumulative_custom.columns:
    axes[0].plot(cumulative_custom.index, cumulative_custom[col], linewidth=2, label=col)
axes[0].axhline(y=1, color='black', linestyle='--', alpha=0.3)
axes[0].set_ylabel('Cumulative Return')
axes[0].set_title('Custom Factor Performance')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rolling Sharpe ratios
rolling_window = 63  # ~3 months
for col in custom_factors.columns:
    rolling_sharpe = (
        custom_factors[col].rolling(window=rolling_window).mean() /
        custom_factors[col].rolling(window=rolling_window).std() *
        np.sqrt(252)
    )
    axes[1].plot(custom_factors.index, rolling_sharpe, linewidth=1.5, label=col)
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[1].set_ylabel('Rolling Sharpe Ratio')
axes[1].set_title(f'{rolling_window}-Day Rolling Sharpe Ratio')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Factor correlation
factor_corr = custom_factors.corr()
sns.heatmap(factor_corr, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
           square=True, ax=axes[2], cbar_kws={'label': 'Correlation'})
axes[2].set_title('Factor Correlation Matrix')

plt.tight_layout()
plt.show()

# Calculate factor statistics
print("\n=== Custom Factor Statistics ===")
for col in custom_factors.columns:
    factor_ret = custom_factors[col].dropna()
    total_return = (1 + factor_ret).prod() - 1
    annualized_return = (1 + total_return) ** (252 / len(factor_ret)) - 1
    volatility = factor_ret.std() * np.sqrt(252)
    sharpe = annualized_return / volatility if volatility > 0 else 0
    
    print(f"\n{col}:")
    print(f"  Annualized Return: {annualized_return*100:.2f}%")
    print(f"  Annualized Volatility: {volatility*100:.2f}%")
    print(f"  Sharpe Ratio: {sharpe:.2f}")
    print(f"  Total Return: {total_return*100:.2f}%")

## 5. Factor Combination & Portfolio Optimization

In [ ]:
def optimize_factor_portfolio(factor_returns, target_return=None):
    """Optimize factor weights using mean-variance optimization"""
    n_factors = factor_returns.shape[1]
    
    # Calculate mean returns and covariance
    mean_returns = factor_returns.mean() * 252  # Annualized
    cov_matrix = factor_returns.cov() * 252  # Annualized
    
    # Objective: minimize portfolio variance
    def portfolio_variance(weights):
        return weights @ cov_matrix @ weights
    
    # Constraints
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]  # Weights sum to 1
    
    if target_return is not None:
        constraints.append({
            'type': 'eq',
            'fun': lambda w: w @ mean_returns - target_return
        })
    
    # Bounds: allow short positions
    bounds = tuple((-1, 1) for _ in range(n_factors))
    
    # Initial guess: equal weights
    x0 = np.ones(n_factors) / n_factors
    
    # Optimize
    result = minimize(
        portfolio_variance,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    
    return result.x

# Combine original and custom factors
all_factors = pd.concat([factors, custom_factors], axis=1).dropna()

# Optimize portfolio
optimal_weights = optimize_factor_portfolio(all_factors)

# Calculate portfolio returns
portfolio_returns = (all_factors * optimal_weights).sum(axis=1)
cumulative_portfolio = (1 + portfolio_returns).cumprod()

# Equal-weight benchmark
equal_weights = np.ones(len(all_factors.columns)) / len(all_factors.columns)
equal_weight_returns = (all_factors * equal_weights).sum(axis=1)
cumulative_equal = (1 + equal_weight_returns).cumprod()

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Optimal weights
factor_names = all_factors.columns
colors = ['green' if w > 0 else 'red' for w in optimal_weights]
axes[0].bar(range(len(optimal_weights)), optimal_weights, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_xticks(range(len(optimal_weights)))
axes[0].set_xticklabels(factor_names, rotation=45, ha='right')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_ylabel('Weight')
axes[0].set_title('Optimal Factor Weights (Mean-Variance Optimization)')
axes[0].grid(True, alpha=0.3, axis='y')

# Cumulative performance
axes[1].plot(cumulative_portfolio.index, cumulative_portfolio, 'b-', linewidth=2.5, label='Optimized Portfolio')
axes[1].plot(cumulative_equal.index, cumulative_equal, 'r--', linewidth=2, label='Equal Weight')
axes[1].axhline(y=1, color='black', linestyle='--', alpha=0.3)
axes[1].set_ylabel('Cumulative Return')
axes[1].set_title('Factor Portfolio Performance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Rolling volatility comparison
rolling_window = 63
rolling_vol_opt = portfolio_returns.rolling(window=rolling_window).std() * np.sqrt(252) * 100
rolling_vol_eq = equal_weight_returns.rolling(window=rolling_window).std() * np.sqrt(252) * 100

axes[2].plot(rolling_vol_opt.index, rolling_vol_opt, 'b-', linewidth=2, label='Optimized')
axes[2].plot(rolling_vol_eq.index, rolling_vol_eq, 'r--', linewidth=2, label='Equal Weight')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Annualized Volatility (%)')
axes[2].set_title(f'{rolling_window}-Day Rolling Volatility')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Performance comparison
def calc_metrics(returns):
    total_ret = (1 + returns).prod() - 1
    ann_ret = (1 + total_ret) ** (252 / len(returns)) - 1
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    return {'Return': ann_ret, 'Volatility': ann_vol, 'Sharpe': sharpe}

metrics_opt = calc_metrics(portfolio_returns)
metrics_eq = calc_metrics(equal_weight_returns)

print("\n=== Portfolio Performance Comparison ===")
print("\nOptimized Portfolio:")
for k, v in metrics_opt.items():
    print(f"  {k}: {v*100:.2f}%" if k != 'Sharpe' else f"  {k}: {v:.2f}")

print("\nEqual Weight Portfolio:")
for k, v in metrics_eq.items():
    print(f"  {k}: {v*100:.2f}%" if k != 'Sharpe' else f"  {k}: {v:.2f}")

## 6. Factor Decay Analysis

In [ ]:
def calculate_factor_decay(returns, factor_signal, max_lag=20):
    """Calculate predictive power decay of a factor over time"""
    correlations = []
    
    for lag in range(1, max_lag + 1):
        # Shift returns by lag
        future_returns = returns.shift(-lag)
        
        # Calculate correlation between signal and future returns
        corr_matrix = factor_signal.corrwith(future_returns)
        avg_corr = corr_matrix.mean()
        
        correlations.append(avg_corr)
    
    return correlations

# Calculate momentum signal (past 60-day returns)
momentum_signal = returns.rolling(window=60).apply(lambda x: (1 + x).prod() - 1)

# Calculate volatility signal
volatility_signal = returns.rolling(window=20).std()

# Calculate factor decay
momentum_decay = calculate_factor_decay(returns, momentum_signal, max_lag=20)
volatility_decay = calculate_factor_decay(returns, -volatility_signal, max_lag=20)  # Negative for low-vol

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Factor decay
lags = range(1, 21)
axes[0].plot(lags, momentum_decay, 'bo-', linewidth=2, markersize=6, label='Momentum')
axes[0].plot(lags, volatility_decay, 'ro-', linewidth=2, markersize=6, label='Low Volatility')
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[0].set_xlabel('Lag (days)')
axes[0].set_ylabel('Average Correlation')
axes[0].set_title('Factor Predictive Power Decay')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Half-life estimation
def estimate_half_life(decay_values):
    """Estimate half-life of factor signal"""
    initial_value = abs(decay_values[0])
    half_value = initial_value / 2
    
    for i, val in enumerate(decay_values):
        if abs(val) <= half_value:
            return i + 1
    return len(decay_values)

momentum_halflife = estimate_half_life(momentum_decay)
volatility_halflife = estimate_half_life(volatility_decay)

# Bar chart comparison
factors_comparison = ['Momentum', 'Low Volatility']
half_lives = [momentum_halflife, volatility_halflife]
initial_corrs = [abs(momentum_decay[0]), abs(volatility_decay[0])]

x = np.arange(len(factors_comparison))
width = 0.35

bars1 = axes[1].bar(x - width/2, half_lives, width, label='Half-life (days)', alpha=0.8)
axes[1].set_ylabel('Half-life (days)')
axes[1].set_title('Factor Signal Half-life Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(factors_comparison)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Add values on bars
for bar in bars1:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}d', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n=== Factor Decay Analysis ===")
print(f"\nMomentum Factor:")
print(f"  Initial correlation: {abs(momentum_decay[0]):.4f}")
print(f"  Half-life: {momentum_halflife} days")
print(f"  20-day correlation: {abs(momentum_decay[-1]):.4f}")

print(f"\nLow Volatility Factor:")
print(f"  Initial correlation: {abs(volatility_decay[0]):.4f}")
print(f"  Half-life: {volatility_halflife} days")
print(f"  20-day correlation: {abs(volatility_decay[-1]):.4f}")

## 7. Summary

### Key Concepts:

1. **Factor Models**
   - Fama-French three-factor model
   - Alpha generation and factor exposures
   - Model fit and residual analysis

2. **Principal Component Analysis**
   - Dimensionality reduction
   - Variance decomposition
   - Factor extraction

3. **Custom Factor Construction**
   - Momentum factors (trend-following)
   - Low volatility factors (defensive)
   - Quality factors (risk-adjusted returns)
   - Cross-sectional ranking

4. **Factor Combination**
   - Mean-variance optimization
   - Multi-factor portfolios
   - Risk reduction through diversification

5. **Factor Decay**
   - Predictive power over time
   - Signal half-life estimation
   - Optimal holding periods

### Applications:
- Portfolio construction
- Risk factor exposure management
- Alpha generation strategies
- Factor timing and allocation
- Performance attribution